Imports

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import math
import plotly.express as px

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [ ]:
airlines_df = pd.read_csv('data/airlines.csv')
airports_df = pd.read_csv('data/airports.csv')
flights_df = pd.read_csv('data/flights.csv')

#filter df to only include June flights
filtered_flights_df = flights_df[flights_df["MONTH"] == 6]

Create date column

In [ ]:
#rename DAY column
filtered_flights_df = filtered_flights_df.rename(columns={'DAY': 'DAY_OF_MONTH'})

#add DATE column
filtered_flights_df["DATE"] = pd.to_datetime({
    "year": filtered_flights_df["YEAR"],
    "month": filtered_flights_df["MONTH"],
    "day": filtered_flights_df["DAY_OF_MONTH"]
})

Set appropraite data types to time-related columns

*Int64 was used where DATETIME would be more intuitive, and float64 was used instead of int64 (there were no values with a number following the decimal point).*

In [ ]:
filtered_flights_df.dtypes

#create function for converting Int64 columns to datetime
def hhmm_to_timedelta(s):
    s = s.astype("Int64")
    hours = s // 100
    minutes = s % 100
    return pd.to_timedelta(hours, unit="h") + pd.to_timedelta(minutes, unit="m")

#specify which columns should be converted to datetime
datetime_cols = [
    "SCHEDULED_DEPARTURE",
    "DEPARTURE_TIME",
    "WHEELS_OFF",
    "WHEELS_ON",
    "SCHEDULED_ARRIVAL",
    "ARRIVAL_TIME"
]

#convert columns to datetime
for c in datetime_cols:
    filtered_flights_df[c] = filtered_flights_df["DATE"] + hhmm_to_timedelta(filtered_flights_df[c])

#specify which columns should be converted to Int64
duration_cols = [
"DEPARTURE_DELAY",
"TAXI_OUT",
"SCHEDULED_TIME",
"ELAPSED_TIME",
"AIR_TIME",
"TAXI_IN",
"ARRIVAL_DELAY"
]

#convert columns to Int64
filtered_flights_df[duration_cols] = filtered_flights_df[duration_cols].astype("Int64")

#drop redundant columns
filtered_flights_df = filtered_flights_df.drop(columns=[
    "YEAR",
    "MONTH",
    "DAY_OF_MONTH",
    "DAY_OF_WEEK",
])

Rename and reorder columns

In [ ]:
rename_map = {

    # departure timeline
    "SCHEDULED_DEPARTURE": "SCHEDULED_DEPARTURE_TIME",
    "DEPARTURE_TIME": "ACTUAL_DEPARTURE_TIME",
    "DEPARTURE_DELAY": "DEPARTURE_DELAY_MINS",
    "TAXI_OUT": "TAXI_OUT_MINS",
    "WHEELS_OFF": "WHEELS_OFF_TIME",

    # flight performance
    "SCHEDULED_TIME": "SCHEDULED_TOTAL_TIME_MINS",
    "ELAPSED_TIME": "ACTUAL_TOTAL_TIME_MINS",
    "AIR_TIME": "ACTUAL_AIR_TIME_MINS",

    # arrival timeline
    "WHEELS_ON": "WHEELS_ON_TIME",
    "TAXI_IN": "TAXI_IN_MINS",
    "SCHEDULED_ARRIVAL": "SCHEDULED_ARRIVAL_TIME",
    "ARRIVAL_TIME": "ACTUAL_ARRIVAL_TIME",
    "ARRIVAL_DELAY": "ARRIVAL_DELAY_MINS"
}

filtered_flights_df = filtered_flights_df.rename(columns=rename_map)

#REORDER

filtered_flights_df = filtered_flights_df[[

    "DATE",

    #flight identity
    "AIRLINE",
    "FLIGHT_NUMBER",
    "TAIL_NUMBER",
    "ORIGIN_AIRPORT",
    "DESTINATION_AIRPORT",
    "DISTANCE",

    #departure timeline
    "SCHEDULED_DEPARTURE_TIME",
    "ACTUAL_DEPARTURE_TIME",
    "DEPARTURE_DELAY_MINS",
    "TAXI_OUT_MINS",
    "WHEELS_OFF_TIME",

    #flight performance
    "SCHEDULED_TOTAL_TIME_MINS",
    "ACTUAL_TOTAL_TIME_MINS",
    "ACTUAL_AIR_TIME_MINS",

    #arrival timeline
    "WHEELS_ON_TIME",
    "TAXI_IN_MINS",
    "SCHEDULED_ARRIVAL_TIME",
    "ACTUAL_ARRIVAL_TIME",
    "ARRIVAL_DELAY_MINS",

    #cancellations
    "DIVERTED",
    "CANCELLED",
    "CANCELLATION_REASON",

    #delays
    "AIR_SYSTEM_DELAY",
    "SECURITY_DELAY",
    "AIRLINE_DELAY",
    "LATE_AIRCRAFT_DELAY",
    "WEATHER_DELAY"

]]

Remove cancelled flights

In [ ]:
before_rows = len(filtered_flights_df)

filtered_flights_df = filtered_flights_df[
    filtered_flights_df["CANCELLED"] != 1
]

filtered_flights_df = filtered_flights_df.drop(columns=[
    "CANCELLED",
    "CANCELLATION_REASON"
])

after_rows = len(filtered_flights_df)

print(f"Rows dropped: {before_rows - after_rows}")

Remove rows with missing departure delay

In [ ]:
before_rows = len(filtered_flights_df)

filtered_flights_df = filtered_flights_df[
    filtered_flights_df["DEPARTURE_DELAY_MINS"].notna()
]

after_rows = len(filtered_flights_df)

print(f"Rows dropped (missing departure delay): {before_rows - after_rows}")

Create new column to flag significant delays

In [ ]:
filtered_flights_df["IS_DELAYED"] = np.where(filtered_flights_df["DEPARTURE_DELAY_MINS"] > 15, 1, 0)

Analysis



1.   What percentage of flights are delayed?



In [ ]:
total = len(filtered_flights_df)
delayed = len(filtered_flights_df[filtered_flights_df["IS_DELAYED"] == 1])
delayed_percent = math.ceil((delayed / total) * 100)

print(f"Percent of flights delayed: {delayed_percent}%")

2.   Which airlines have the worst average departure delays?


In [ ]:
merged_df = filtered_flights_df.merge(airlines_df, left_on="AIRLINE", right_on="IATA_CODE")

result_df = merged_df[["AIRLINE_y", "DEPARTURE_DELAY_MINS"]]

result_df = result_df.rename(columns={"AIRLINE_y": "AIRLINE"})

result = result_df.groupby("AIRLINE")["DEPARTURE_DELAY_MINS"].mean().sort_values(ascending=False).head()

print("Top 5 Airlines by Average Departure Delay:")

for airline, delay in result.items():
    print(f"{airline}: {delay:.2f} minutes")

3.   Which airports have the worst average departure delays?


In [ ]:
merged_df = filtered_flights_df.merge(airports_df, left_on="ORIGIN_AIRPORT", right_on="IATA_CODE")

result_df = merged_df[["AIRPORT", "DEPARTURE_DELAY_MINS"]]

result_df = result_df.rename(columns={"AIRPORT_y": "AIRPORT"})

result = result_df.groupby("AIRPORT")["DEPARTURE_DELAY_MINS"].mean().sort_values(ascending=False).head()

print("Top 5 Airports by Average Departure Delay:")

for airport, delay in result.items():
    print(f"{airport}: {delay:.2f} minutes")

4.   Does departure time affect delays?


In [ ]:
filtered_flights_df["DEPARTURE_HOUR"] = filtered_flights_df["ACTUAL_DEPARTURE_TIME"].dt.hour

hourly_delay = filtered_flights_df.groupby("DEPARTURE_HOUR")["DEPARTURE_DELAY_MINS"].mean().sort_index()

print("Average Departure Delay by Hour:")
for hour, delay in hourly_delay.items():
    print(f"{hour:02d}:00 - {delay:.2f} minutes")


In [ ]:
filtered_flights_df["DEPARTURE_HOUR"] = filtered_flights_df["ACTUAL_DEPARTURE_TIME"].dt.hour

# Hourly average
hourly_delay = (
    filtered_flights_df
    .groupby("DEPARTURE_HOUR", as_index=False)["DEPARTURE_DELAY_MINS"]
    .mean()
)

# Overall 24-hour average
overall_avg = filtered_flights_df["DEPARTURE_DELAY_MINS"].mean()

fig = px.line(
    hourly_delay,
    x="DEPARTURE_HOUR",
    y="DEPARTURE_DELAY_MINS",
    markers=True,
    title="Average Departure Delay by Hour (with 24h Average)"
)

# Add horizontal reference line
fig.add_hline(
    y=overall_avg,
    line_dash="dash",
    line_color="red",
    annotation_text="24h Average",
    annotation_position="top left"
)

fig.update_layout(
    width=1000,
    height=500,
    xaxis=dict(dtick=1)
)

fig.show()

5.   At which time of day are delays worst?


In [ ]:
filtered_flights_df["DEPARTURE_HOUR"] = filtered_flights_df["ACTUAL_DEPARTURE_TIME"].dt.hour

filtered_flights_df["TIME_OF_DAY"] = np.select(
    [
        filtered_flights_df["DEPARTURE_HOUR"].between(5, 11),
        filtered_flights_df["DEPARTURE_HOUR"].between(12, 16),
        filtered_flights_df["DEPARTURE_HOUR"].between(17, 21),
    ],
    ["Morning", "Afternoon", "Evening"],
    default="Night"
)

In [ ]:
sns.boxplot(
    data=filtered_flights_df,
    x="TIME_OF_DAY",
    y="DEPARTURE_DELAY_MINS",
    order=["Morning", "Afternoon", "Evening", "Night"]
)

In [ ]:
# Aggregations
summary = filtered_flights_df.groupby("TIME_OF_DAY").agg(
    avg_delay=("DEPARTURE_DELAY_MINS", "mean"),
    flight_count=("DEPARTURE_DELAY_MINS", "count")
).reindex(["Morning", "Afternoon", "Evening", "Night"])

fig, ax1 = plt.subplots(figsize=(10, 6))

# Bars = average delay
ax1.bar(summary.index, summary["avg_delay"], color="steelblue", alpha=0.7)
ax1.set_ylabel("Average Delay (minutes)", color="steelblue")
ax1.set_title("Delay vs Flight Volume by Time of Day")

# Second axis = flight count
ax2 = ax1.twinx()
ax2.plot(summary.index, summary["flight_count"], color="darkred", marker="o")
ax2.set_ylabel("Number of Flights", color="darkred")

plt.show()

As we can see there is an inverse relationship between flight volume and number of delays.

In [ ]:
delay_reasons = [
    "AIR_SYSTEM_DELAY",
    "SECURITY_DELAY",
    "AIRLINE_DELAY",
    "LATE_AIRCRAFT_DELAY",
    "WEATHER_DELAY"
]

reason_totals = filtered_flights_df[delay_reasons].sum().sort_values(ascending=False)
reason_pct = (reason_totals / reason_totals.sum()) * 100

In [ ]:
plt.figure(figsize=(10,5))

ax = sns.barplot(
    x=reason_totals.index,
    y=reason_totals.values
)

plt.title("Total Delay Minutes by Cause")
plt.xlabel("Cause")
plt.ylabel("Total Delay (minutes)")
plt.xticks(rotation=30)

# Add % labels
for i, v in enumerate(reason_totals.values):
    plt.text(
        i,
        v,
        f"{reason_pct.iloc[i]:.1f}%",
        ha="center",
        va="bottom"
    )

plt.show()

In [ ]:
df_melted = filtered_flights_df.melt(
    id_vars=["TIME_OF_DAY"],
    value_vars=delay_reasons,
    var_name="REASON",
    value_name="DELAY"
)

summary = df_melted.groupby(["TIME_OF_DAY", "REASON"])["DELAY"].sum().reset_index()

plt.figure(figsize=(12,6))

sns.barplot(
    data=summary,
    x="TIME_OF_DAY",
    y="DELAY",
    hue="REASON",
    order=["Morning", "Afternoon", "Evening", "Night"]
)

plt.title("Delay Causes by Time of Day")
plt.ylabel("Total Delay Minutes")
plt.xlabel("Time of Day")

plt.show()